In [8]:
# 0. IMPORT LIBRARIES
# basic data handling
import pandas as pd
import numpy as np

# for splitting dataset
from sklearn.model_selection import train_test_split

# for file path
import os

# 1. LOAD DATA
# define path to raw dataset
# adjust path based on your project structure
DATA_PATH = "/content/data/raw/Cleaned_E_Commerce_Dataset.csv"

# read csv file into dataframe
df = pd.read_csv(DATA_PATH)

# show basic info
print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (5630, 20)
   CustomerID  Churn  Tenure PreferredLoginDevice  CityTier  WarehouseToHome  \
0       50001      1     4.0         Mobile Phone         3              6.0   
1       50002      1     9.0         Mobile Phone         1              8.0   
2       50003      1     9.0         Mobile Phone         1             30.0   
3       50004      1     0.0         Mobile Phone         3             15.0   
4       50005      1     0.0         Mobile Phone         1             12.0   

  PreferredPaymentMode  Gender  HourSpendOnApp  NumberOfDeviceRegistered  \
0           Debit Card  Female             3.0                         3   
1                  UPI    Male             3.0                         4   
2           Debit Card    Male             2.0                         4   
3           Debit Card    Male             2.0                         4   
4          Credit Card    Male             3.0                         3   

     PreferedOrderCat  SatisfactionS

In [9]:
# 2. DROP CUSTOMER ID
# CustomerID is unique identifier
# model can memorize it -> fake performance (overfitting)
# so we must remove it

if "CustomerID" in df.columns:
    df = df.drop("CustomerID", axis=1)
    print("CustomerID dropped")
else:
    print("CustomerID not found (skip)")

CustomerID dropped


In [10]:
# 3. DEFINE FEATURES & TARGET
# target variable (churn label)
TARGET = "Churn"

# X = all features except target
X = df.drop(TARGET, axis=1)

# y = target
y = df[TARGET]

# check shape
print("X shape:", X.shape)
print("y distribution:")
print(y.value_counts())

X shape: (5630, 18)
y distribution:
Churn
0    4682
1     948
Name: count, dtype: int64


In [11]:
# 4. TRAIN-TEST SPLIT (CRITICAL)
# split dataset into train and test
# test_size = 0.2 -> 20% test, 80% train
# stratify=y -> keep same class distribution (important for imbalance)
# random_state=42 -> reproducible results

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# check result
print("\nSplit done:")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("\nTrain distribution:")
print(y_train.value_counts())

print("\nTest distribution:")
print(y_test.value_counts())


Split done:
X_train: (4504, 18)
X_test : (1126, 18)

Train distribution:
Churn
0    3746
1     758
Name: count, dtype: int64

Test distribution:
Churn
0    936
1    190
Name: count, dtype: int64


In [12]:
# 5. FREEZE TEST SET (VERY IMPORTANT)
# from this point:
# - NEVER use test set for training
# - NEVER fit anything on test set
# - ONLY use test set for final evaluation

print("\nTest set is now FROZEN. No more touching until evaluation.")


Test set is now FROZEN. No more touching until evaluation.


In [13]:
# 6. SAVE PROCESSED DATA
# create processed data folder if not exists
os.makedirs("data/processed", exist_ok=True)

# save to csv files
X_train.to_csv("data/processed/X_train.csv", index=False)
y_train.to_csv("data/processed/y_train.csv", index=False)

X_test.to_csv("data/processed/X_test.csv", index=False)
y_test.to_csv("data/processed/y_test.csv", index=False)

print("\nSaved successfully:")
print(" - data/processed/X_train.csv")
print(" - data/processed/y_train.csv")
print(" - data/processed/X_test.csv")
print(" - data/processed/y_test.csv")


Saved successfully:
 - data/processed/X_train.csv
 - data/processed/y_train.csv
 - data/processed/X_test.csv
 - data/processed/y_test.csv


### **Data Splitting & Initial Cleaning**

**1. Early Data Splitting (Preventing Data Leakage)**
* **Action:** We split the data into Training and Testing sets *before* applying any preprocessing steps.
* **Reason:** If we preprocess the entire dataset together, the process will "learn" information from the test set. This causes **Data Leakage** (the model basically sees the future), leading to fake high scores. Splitting the data first is the #1 rule in building a Machine Learning pipeline.

**2. Using Stratified Split (`stratify=y`)**
* **Action:** We applied a stratified split instead of a completely random split.
* **Reason:** Customer churn data is usually imbalanced. A random split could create different distributions (e.g., 5% churn in the Train set but 20% in the Test set), which leads to wrong evaluations. Stratifying ensures that both sets have the exact same ratio of the target variable.

**3. Dropping the CustomerID Column**
* **Action:** We removed the unique identifier (CustomerID) from the dataset.
* **Reason:** Customer IDs are unique to each person and carry no real behavioral meaning. If kept, the model might just "memorize" the IDs to predict the outcome, creating a fake perfect score (like an ROC AUC of 1.0). Dropping it ensures the model learns actual patterns and can generalize to new data.